# Incoorporating new files

## Setup

Add your data path here (`base_path`)

In [31]:
# Add path
import sys
sys.path.append('../')

import importlib 
from util import config
from util import utilities as util
import util.data_loader as data_loader  # <-- add
from util.data_loader import DataLoader

import os
import shutil
import pandas as pd
from tqdm import tqdm

# -------------------------
# NEW DATA ROOT (only)
# -------------------------
NEW_ROOT = "/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/new_videos"

# point metadata CSVs to NEW only
config.trial_info_path  = os.path.join(NEW_ROOT, "trial_info_new_animals.csv")
config.animal_info_path = os.path.join(NEW_ROOT, "animal_info_20260427.csv")

# point data home to NEW only (important if DataLoader uses config.data_path)
config.data_path = NEW_ROOT

# reload util so it reads the updated config values
importlib.reload(util)
importlib.reload(data_loader)

# now this can ONLY read the NEW csvs
metadata_new = util.load_metadata(type="combined")

print("Using:")
print(" trial_info_path :", config.trial_info_path)
print(" animal_info_path:", config.animal_info_path)
print(" data_path       :", config.data_path)
print(" rows            :", len(metadata_new))

Using:
 trial_info_path : /Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/new_videos/trial_info_new_animals.csv
 animal_info_path: /Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/new_videos/animal_info_20260427.csv
 data_path       : /Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/new_videos
 rows            : 1614


## Utilities

In [32]:
def load_new_metadata():
    # util.load_metadata reads config.animal_info_path / config.trial_info_path
    return util.load_metadata(type="combined")

base_path = NEW_ROOT

def get_date_format(date):
    ''' Convert YYYYMMDD to "YYYY-MM-DD" '''
    date_str = str(date)
    date_formatted = f"{date_str[0:4]}-{date_str[4:6]}-{date_str[6:8]}"
    return date_formatted

def generate_source_file_path(dl, data_type = 'behVideo'):
    '''
    Files for each trial:
        * trackInfo: `*.positions.csv`
        * exampleImg: `*.trace.png`
        * behVideo: `*.avi`
    Files for each day:
        * triggerLoc: `*.locations.csv`
    '''
    date_formatted = get_date_format(dl.date)
    date_folder_path = os.path.join(base_path, 'data', date_formatted)

    if data_type == 'triggerLoc':
        return os.path.join(date_folder_path, 'locations.csv')
    elif data_type == 'behVideo':
        return os.path.join(date_folder_path, 'movie', f"{date_formatted}_{dl.id}_trial_{dl.trial_id}.avi")
    elif data_type == 'trackInfo':
        return os.path.join(date_folder_path, 'movie', 'tracking', f"{date_formatted}_{dl.id}_trial_{dl.trial_id}_positions.csv")
    elif data_type == 'exampleImg':
        return os.path.join(date_folder_path, 'movie', 'tracking', f"{date_formatted}_{dl.id}_trial_{dl.trial_id}_trace.png")
    else:
        print('Unknown data type')

def generate_desti_file_path(dl, data_type = 'behVideo'):
    '''
    Files for each trial:
        * trackInfo: `*.positions.csv`
        * exampleImg: `*.trace.png`
        * behVideo: `*.avi`
    Files for each day:
        * triggerLoc: `*.locations.csv`
    '''
    if data_type == 'triggerLoc':
        return dl.generate_filepath('triggerLoc', 'rawdata', 'behav', 'csv')
    if data_type == 'trackInfo':
        return dl.generate_filepath('position', 'derivatives', 'behav', 'csv')
    if data_type == 'exampleImg':
        return dl.generate_filepath('trace', 'derivatives', 'behav', 'png')
    if data_type == 'behVideo':
        return dl.generate_filepath('video', 'rawdata', 'behav', 'avi')
    else: 
        print('Unknown data type')

def check_source_file_exists(dl, data_type):
    '''
    Files for each trial:
        * trackInfo: `*.positions.csv`
        * exampleImg: `*.trace.png`
        * behVideo: `*.avi`
    Files for each day:
        * triggerLoc: `*.locations.csv`
    '''
    file = generate_source_file_path(dl, data_type)
    if not os.path.isfile(file):
        print(f'{dl.id}-{dl.date}-trial_{dl.trial_id}: {data_type} missing')
        return False
    return True


In [33]:
metadata = load_new_metadata()

for col in ['sub', 'ses', 'trial_id', 'date', 'trial_type_day']:
    if col in metadata.columns:
        metadata[col] = pd.to_numeric(metadata[col], errors='coerce').astype('Int64')

metadata.head()

,sub,id,date,ses,trial_id,trial_type,trial_type_day,test_type,usable,trial_note,genotype,test_order,remapping,short_duration,LFP,animal_note
0,64,N1,20260316,1,1,H,1,NaN,1,NaN,NAc,NaN,0,0,0,NaN
1,64,N1,20260316,1,2,H,1,NaN,1,NaN,NAc,NaN,0,0,0,NaN
2,64,N1,20260316,1,3,H,1,NaN,1,NaN,NAc,NaN,0,0,0,NaN
3,64,N1,20260316,1,4,H,1,NaN,1,NaN,NAc,NaN,0,0,0,NaN
4,64,N1,20260316,1,5,H,1,NaN,1,NaN,NAc,NaN,0,0,0,NaN


## Check existance of all files  

Files for each trial:  
    * trackInfo: `*.positions.csv`  
    * exampleImg: `*.trace.png`  
    * behVideo: `*.avi`  
Files for each day:  
    * triggerLoc: `*.locations.csv`  

In [35]:
data_types = {'trackInfo', 'exampleImg', 'behVideo', 'triggerLoc'}

flag = True
for rowIdx in range(len(metadata)):
    dl = DataLoader(metadata_ses=metadata.iloc[rowIdx])
    for data_type in data_types:
        if not check_source_file_exists(dl, data_type):
            flag = False

if flag:
    print('All source files exist!')

N3-20260323-trial_8: trackInfo missing
N3-20260323-trial_8: exampleImg missing
N6-20260319-trial_6: behVideo missing
N6-20260319-trial_6: trackInfo missing
N6-20260319-trial_6: exampleImg missing
C1-20260326-trial_6: trackInfo missing
C1-20260326-trial_6: exampleImg missing
C7-20260326-trial_6: trackInfo missing
C7-20260326-trial_6: exampleImg missing


## Move data

In [36]:
for rowIdx in tqdm(range(len(metadata))):
    dl = DataLoader(metadata_ses=metadata.iloc[rowIdx])
    for data_type in data_types:
        source_file_path = generate_source_file_path(dl, data_type)
        desti_file_path = generate_desti_file_path(dl, data_type)

        os.makedirs(os.path.dirname(desti_file_path), exist_ok=True)

        if not os.path.isfile(source_file_path):
            print(f'{dl.id}-{dl.date}-trial_{dl.trial_id}: {data_type} missing')
            continue

        if data_type == 'triggerLoc':
            loc = pd.read_csv(source_file_path)
            loc = loc.rename(columns={
                'Animal': 'id',
                'Well_row': 'well_row',
                'Well_col': 'well_col',
                'Valence': 'valence',
            })
            loc = loc[loc['id'] == dl.id].reset_index(drop=True)
            loc.to_csv(desti_file_path, index=False)
        else:
            shutil.copy2(source_file_path, desti_file_path)
        

 12%|█▏        | 194/1614 [00:02<00:17, 81.30it/s]

N3-20260323-trial_8: trackInfo missing
N3-20260323-trial_8: exampleImg missing


 21%|██▏       | 345/1614 [00:04<00:13, 94.81it/s]

N6-20260319-trial_6: behVideo missing
N6-20260319-trial_6: trackInfo missing
N6-20260319-trial_6: exampleImg missing


 28%|██▊       | 448/1614 [00:05<00:14, 82.73it/s]

C1-20260326-trial_6: trackInfo missing
C1-20260326-trial_6: exampleImg missing


 51%|█████     | 822/1614 [00:09<00:10, 78.82it/s]

C7-20260326-trial_6: trackInfo missing
C7-20260326-trial_6: exampleImg missing


100%|██████████| 1614/1614 [00:19<00:00, 83.69it/s]
